# 02 — Silver Transform | RHC Training Analytics

Este notebook implementa a camada **Silver** do projeto. A entrada é a execução Bronze mais recente armazenada no Google Drive.

Objetivos: validar qualidade, padronizar tipos, tratar valores textuais/nulos, identificar duplicidades, validar relacionamentos e preparar datasets confiáveis para a futura camada Gold.

Fluxo: `Bronze Parquet → Data Quality → Transformações técnicas → Silver Parquet`.


## 1. Montar Google Drive e configurar caminhos


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import re
import pandas as pd

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 200)

BASE_DIR = Path('/content/drive/MyDrive/rhc-training-analytics/data')
BRONZE_ROOT = BASE_DIR / 'bronze'
SILVER_ROOT = BASE_DIR / 'silver'
RUN_TS = datetime.now(timezone.utc)
RUN_ID = RUN_TS.strftime('%Y%m%dT%H%M%SZ')

bronze_runs = sorted(BRONZE_ROOT.glob('extract_date=*/run_id=*'))
if not bronze_runs:
    raise FileNotFoundError('Nenhuma execução Bronze encontrada.')

BRONZE_DIR = bronze_runs[-1]
SILVER_DIR = SILVER_ROOT / f'process_date={RUN_TS:%Y-%m-%d}' / f'run_id={RUN_ID}'
SILVER_DIR.mkdir(parents=True, exist_ok=True)

print('Bronze selecionada:', BRONZE_DIR)
print('Silver destino:', SILVER_DIR)


## 2. Carregar datasets Bronze


In [ ]:
TABLES = [
    'profiles', 'body_measurements', 'workout_sessions', 'workout_exercises',
    'exercise_catalog', 'exercise_records', 'training_programs', 'program_phases',
    'program_sessions', 'program_exercises', 'program_enrollments',
    'program_exercise_exposures',
]

data = {table: pd.read_parquet(BRONZE_DIR / f'{table}.parquet') for table in TABLES}

pd.DataFrame([
    {'table': name, 'rows': len(df), 'columns': len(df.columns)}
    for name, df in data.items()
]).sort_values('table')


## 3. Data profiling antes das transformações

Esta etapa mede nulos, cardinalidade, duplicidade de linhas e tipos inferidos antes de qualquer alteração.


In [ ]:
profile_records = []
table_quality = []

for table, df in data.items():
    table_quality.append({
        'table': table,
        'rows': len(df),
        'columns': len(df.columns),
        'duplicate_rows': int(df.duplicated().sum()),
    })
    for col in df.columns:
        s = df[col]
        profile_records.append({
            'table': table,
            'column': col,
            'dtype_before': str(s.dtype),
            'null_count': int(s.isna().sum()),
            'null_pct': round(float(s.isna().mean() * 100), 2),
            'distinct_count': int(s.nunique(dropna=True)),
        })

profile_before_df = pd.DataFrame(profile_records)
table_quality_df = pd.DataFrame(table_quality)
display(table_quality_df)
display(profile_before_df.sort_values(['null_pct'], ascending=False).head(40))


## 4. Funções de padronização técnica

As regras abaixo são conservadoras: não removem registros por nulos de negócio e não alteram métricas. A Silver padroniza representação e tipos.


In [ ]:
NULL_STRINGS = {'', 'null', 'none', 'nan', 'nat'}
DATE_NAME_RE = re.compile(r'(^|_)(date|created_at|updated_at|started_at|finished_at|completed_at|enrolled_at|ended_at|measured_at|performed_at|recorded_at)$')
BOOL_TRUE = {'true', 't', '1', 'yes', 'y'}
BOOL_FALSE = {'false', 'f', '0', 'no', 'n'}

def clean_object_series(s: pd.Series) -> pd.Series:
    if s.dtype != 'object':
        return s
    out = s.map(lambda x: x.strip() if isinstance(x, str) else x)
    return out.map(lambda x: pd.NA if isinstance(x, str) and x.lower() in NULL_STRINGS else x)

def maybe_boolean(s: pd.Series) -> pd.Series:
    if s.dtype != 'object':
        return s
    vals = set(s.dropna().astype(str).str.lower().unique())
    if vals and vals.issubset(BOOL_TRUE | BOOL_FALSE):
        mapping = {**{v: True for v in BOOL_TRUE}, **{v: False for v in BOOL_FALSE}}
        return s.map(lambda x: mapping.get(str(x).lower()) if pd.notna(x) else pd.NA).astype('boolean')
    return s

def transform_table(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [c.strip().lower() for c in out.columns]
    for col in out.columns:
        out[col] = clean_object_series(out[col])
        out[col] = maybe_boolean(out[col])
        if DATE_NAME_RE.search(col):
            parsed = pd.to_datetime(out[col], errors='coerce', utc=True)
            # Só converte quando valores não nulos foram reconhecidos como data.
            if out[col].notna().sum() == 0 or parsed.notna().sum() > 0:
                out[col] = parsed
    return out


## 5. Aplicar transformações Silver


In [ ]:
silver = {table: transform_table(df) for table, df in data.items()}

transform_summary = []
for table in TABLES:
    before = data[table]
    after = silver[table]
    transform_summary.append({
        'table': table,
        'rows_before': len(before),
        'rows_after': len(after),
        'columns_before': len(before.columns),
        'columns_after': len(after.columns),
    })
transform_summary_df = pd.DataFrame(transform_summary)
transform_summary_df


## 6. Diagnóstico de chaves e duplicidades

Não removemos duplicidades automaticamente. Primeiro identificamos possíveis chaves (`id`) e registramos qualquer violação para análise.


In [ ]:
key_checks = []
for table, df in silver.items():
    if 'id' in df.columns:
        key_checks.append({
            'table': table,
            'key': 'id',
            'null_keys': int(df['id'].isna().sum()),
            'duplicate_keys': int(df['id'].duplicated().sum()),
            'unique_keys': int(df['id'].nunique(dropna=True)),
        })
key_checks_df = pd.DataFrame(key_checks)
key_checks_df


## 7. Diagnóstico de integridade referencial

As relações abaixo representam o núcleo analítico conhecido do domínio. O objetivo aqui é identificar chaves órfãs antes da modelagem Gold.


In [ ]:
RELATIONSHIPS = [
    ('body_measurements', 'profile_id', 'profiles', 'id'),
    ('workout_sessions', 'profile_id', 'profiles', 'id'),
    ('workout_exercises', 'workout_session_id', 'workout_sessions', 'id'),
    ('workout_exercises', 'exercise_id', 'exercise_catalog', 'id'),
    ('exercise_records', 'profile_id', 'profiles', 'id'),
    ('exercise_records', 'exercise_id', 'exercise_catalog', 'id'),
    ('program_phases', 'program_id', 'training_programs', 'id'),
    ('program_sessions', 'phase_id', 'program_phases', 'id'),
    ('program_exercises', 'session_id', 'program_sessions', 'id'),
    ('program_enrollments', 'program_id', 'training_programs', 'id'),
    ('program_enrollments', 'profile_id', 'profiles', 'id'),
]

relationship_checks = []
for child, fk, parent, pk in RELATIONSHIPS:
    if fk not in silver[child].columns or pk not in silver[parent].columns:
        relationship_checks.append({
            'child_table': child, 'foreign_key': fk, 'parent_table': parent,
            'status': 'column_not_found', 'orphan_rows': None,
        })
        continue
    child_values = silver[child][fk].dropna()
    parent_values = set(silver[parent][pk].dropna())
    orphan_mask = ~child_values.isin(parent_values)
    relationship_checks.append({
        'child_table': child, 'foreign_key': fk, 'parent_table': parent,
        'status': 'checked', 'orphan_rows': int(orphan_mask.sum()),
    })

relationship_checks_df = pd.DataFrame(relationship_checks)
relationship_checks_df


## 8. Identificar colunas JSON / semiestruturadas

Nesta etapa apenas detectamos estruturas JSON. A explosão de arrays será feita de forma explícita quando definirmos a granularidade analítica da Gold.


In [ ]:
def looks_like_json(value) -> bool:
    if not isinstance(value, str):
        return False
    value = value.strip()
    if not (value.startswith('{') or value.startswith('[')):
        return False
    try:
        json.loads(value)
        return True
    except Exception:
        return False

json_columns = []
for table, df in silver.items():
    for col in df.select_dtypes(include='object').columns:
        non_null = df[col].dropna()
        if len(non_null) and non_null.head(50).map(looks_like_json).mean() >= 0.8:
            json_columns.append({
                'table': table, 'column': col,
                'sample_count': min(len(non_null), 50),
            })
json_columns_df = pd.DataFrame(json_columns)
json_columns_df


## 9. Persistir camada Silver e auditoria


In [ ]:
for table, df in silver.items():
    df.to_parquet(SILVER_DIR / f'{table}.parquet', index=False)

profile_before_df.to_parquet(SILVER_DIR / '_profile_before.parquet', index=False)
table_quality_df.to_parquet(SILVER_DIR / '_table_quality.parquet', index=False)
transform_summary_df.to_parquet(SILVER_DIR / '_transform_summary.parquet', index=False)
key_checks_df.to_parquet(SILVER_DIR / '_key_checks.parquet', index=False)
relationship_checks_df.to_parquet(SILVER_DIR / '_relationship_checks.parquet', index=False)
json_columns_df.to_parquet(SILVER_DIR / '_json_columns.parquet', index=False)

manifest = {
    'project': 'rhc-training-analytics',
    'layer': 'silver',
    'run_id': RUN_ID,
    'processed_at_utc': RUN_TS.isoformat(),
    'bronze_source': str(BRONZE_DIR),
    'silver_output': str(SILVER_DIR),
    'tables': TABLES,
    'rules': [
        'normalize column names',
        'trim strings',
        'normalize textual nulls',
        'infer boolean representations',
        'parse timestamp/date columns to UTC when recognized',
        'preserve row granularity',
        'audit primary-key candidates and relationships',
        'detect semistructured JSON columns',
    ],
}
with (SILVER_DIR / '_manifest.json').open('w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print('Silver persistida em:', SILVER_DIR)


## 10. Critérios de sucesso e resumo


In [ ]:
assert all(len(data[t]) == len(silver[t]) for t in TABLES), 'A Silver alterou a granularidade de alguma tabela.'

bad_keys = key_checks_df[(key_checks_df['null_keys'] > 0) | (key_checks_df['duplicate_keys'] > 0)]
orphans = relationship_checks_df[
    (relationship_checks_df['status'] == 'checked') &
    (relationship_checks_df['orphan_rows'].fillna(0) > 0)
]

print(f'✅ Silver processada: {len(TABLES)} tabelas.')
print(f'Problemas em chaves candidatas: {len(bad_keys)}')
print(f'Relacionamentos com registros órfãos: {len(orphans)}')
print(f'Colunas JSON detectadas: {len(json_columns_df)}')
print('Próximo passo: revisar os diagnósticos antes de modelar 03_gold_model.ipynb.')

display(bad_keys)
display(orphans)
display(json_columns_df)
